# データ保存

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# 必要なデータ
- 軌道データ (SM座標系)
- 電子 (omni flux, PA) (~10 keV)
- proton (omni flux, PA) (~ 1 keV)
- 数密度 (Arase: LEP-e & HFA, THEMIS: mom)
- 温度 (Arase: ion平均と電子, THEMIS: proton想定)
- $V_{\mathrm{sys}\perp}$ (Arase: ion平均考慮)
- $B_{\mathrm{tot}}$
- $B_{x}$
- $B_{y}$
- $B_{z}$
- $S_{z}, \bf{S}_{\perp}$

# 軌道データの作成

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import pytplot as pt
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr

pt.del_data('*')

trange = ['2022-09-01/22:25', '2022-09-01/23:15']

earth_radius = 6378.1  # km

psp.projects.themis.state(trange=trange, probe='a')
psp.cotrans(name_in='tha_pos_gse', name_out='tha_pos_sm', coord_in='gse', coord_out='sm')   # 'tha_pos_sm'

themis_a_pos_sm = psp.get_data('tha_pos_sm', xarray=True)
themis_a_pos_sm.values = themis_a_pos_sm.values / earth_radius  # convert to RE
themis_a_pos_sm.attrs['Units'] = 'R_E'
# trangeに合わせてデータを切り出し
themis_a_pos_sm = themis_a_pos_sm.sel(time=slice(trange[0], trange[1]))

print(themis_a_pos_sm)

ergpy.orb(trange=trange, level='l2', datatype='def')  # 'erg_orb_l2_pos_sm'
Arase_pos_sm = psp.get_data('erg_orb_l2_pos_sm', xarray=True)
Arase_pos_sm = Arase_pos_sm.sel(time=slice(trange[0], trange[1]))

print(Arase_pos_sm)

In [ ]:
Arase_pos_sm_x, Arase_pos_sm_y, Arase_pos_sm_z = Arase_pos_sm.values[:,0], Arase_pos_sm.values[:,1], Arase_pos_sm.values[:,2]
Arase_pos_sm_time = Arase_pos_sm.time
themis_a_pos_sm_x, themis_a_pos_sm_y, themis_a_pos_sm_z = themis_a_pos_sm.values[:,0], themis_a_pos_sm.values[:,1], themis_a_pos_sm.values[:,2]
themis_a_pos_sm_time = themis_a_pos_sm.time

In [ ]:
Arase_rmlatmlt_R = np.sqrt(Arase_pos_sm_x**2E0 + Arase_pos_sm_y**2E0 + Arase_pos_sm_z**2E0)
Arase_rmlatmlt_MLAT = np.rad2deg(np.arctan2(Arase_pos_sm_z, np.sqrt(Arase_pos_sm_x**2E0 + Arase_pos_sm_y**2E0)))
Arase_rmlatmlt_MLT = np.rad2deg(np.arctan2(Arase_pos_sm_y, Arase_pos_sm_x)) / 15. + 12.

Arase_rmlatmlt_L    = Arase_rmlatmlt_R / np.cos(np.deg2rad(Arase_rmlatmlt_MLAT))**2E0
Arase_rmlatmlt_L_x  = Arase_rmlatmlt_L / np.sqrt(Arase_pos_sm_x**2E0 + Arase_pos_sm_y**2E0) * Arase_pos_sm_x
Arase_rmlatmlt_L_y  = Arase_rmlatmlt_L / np.sqrt(Arase_pos_sm_x**2E0 + Arase_pos_sm_y**2E0) * Arase_pos_sm_y

THA_rmlatmlt_R = np.sqrt(themis_a_pos_sm_x**2E0 + themis_a_pos_sm_y**2E0 + themis_a_pos_sm_z**2E0)
THA_rmlatmlt_MLAT = np.rad2deg(np.arctan2(themis_a_pos_sm_z, np.sqrt(themis_a_pos_sm_x**2E0 + themis_a_pos_sm_y**2E0)))
THA_rmlatmlt_MLT = np.rad2deg(np.arctan2(themis_a_pos_sm_y, themis_a_pos_sm_x)) / 15. + 12.

THA_rmlatmlt_L    = THA_rmlatmlt_R / np.cos(np.deg2rad(THA_rmlatmlt_MLAT))**2E0
THA_rmlatmlt_L_x  = THA_rmlatmlt_L / np.sqrt(themis_a_pos_sm_x**2E0 + themis_a_pos_sm_y**2E0) * themis_a_pos_sm_x
THA_rmlatmlt_L_y  = THA_rmlatmlt_L / np.sqrt(themis_a_pos_sm_x**2E0 + themis_a_pos_sm_y**2E0) * themis_a_pos_sm_y

In [ ]:
np.median(THA_rmlatmlt_R)

In [ ]:
# 文字の大きさ
mpl.rcParams['font.size'] = 25

In [ ]:
def pretty(ax, xlabel, ylabel):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.minorticks_on()
    ax.grid(which='both', linestyle='--', alpha=0.7)
    ax.axhline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.axvline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.set_aspect('equal', 'box')

fig = plt.figure(figsize=(11, 11), dpi=300)

ax1 = fig.add_subplot(221)
ax1.plot(themis_a_pos_sm_x, themis_a_pos_sm_y, color='magenta', label='THEMIS-A', linewidth=3)
ax1.scatter(themis_a_pos_sm_x[0],  themis_a_pos_sm_y[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax1.scatter(themis_a_pos_sm_x[-1], themis_a_pos_sm_y[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax1.plot(Arase_pos_sm_x, Arase_pos_sm_y, color='green', label='Arase', linewidth=3)
ax1.scatter(Arase_pos_sm_x[0],  Arase_pos_sm_y[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax1.scatter(Arase_pos_sm_x[-1], Arase_pos_sm_y[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax1, 'X‒SM\n'+ r'[$R_{\mathrm{E}}$]', 'Y‒SM\n'+ r'[$R_{\mathrm{E}}$]')
ax1.set_xlim(1, -9)
ax1.set_ylim(8.5, -1.5)
ax1.set_title('(X, Y)')


ax2 = fig.add_subplot(222)
ax2.plot(themis_a_pos_sm_x, themis_a_pos_sm_z, color='magenta', label='THEMIS-A', linewidth=3)
ax2.scatter(themis_a_pos_sm_x[0],  themis_a_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax2.scatter(themis_a_pos_sm_x[-1], themis_a_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax2.plot(Arase_pos_sm_x, Arase_pos_sm_z, color='green', label='Arase', linewidth=3)
ax2.scatter(Arase_pos_sm_x[0],  Arase_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax2.scatter(Arase_pos_sm_x[-1], Arase_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax2, 'X‒SM\n'+ r'[$R_{\mathrm{E}}$]', 'Z‒SM\n'+ r'[$R_{\mathrm{E}}$]')
ax2.set_xlim(1, -9)
ax2.set_ylim(-3, 7)
ax2.set_title('(X, Z)')

ax3 = fig.add_subplot(223)
ax3.plot(themis_a_pos_sm_y, themis_a_pos_sm_z, color='magenta', label='THEMIS-A', linewidth=3)
ax3.scatter(themis_a_pos_sm_y[0],  themis_a_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax3.scatter(themis_a_pos_sm_y[-1], themis_a_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax3.plot(Arase_pos_sm_y, Arase_pos_sm_z, color='green', label='Arase', linewidth=3)
ax3.scatter(Arase_pos_sm_y[0],  Arase_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax3.scatter(Arase_pos_sm_y[-1], Arase_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax3, 'Y‒SM\n'+ r'[$R_{\mathrm{E}}$]', 'Z‒SM\n'+ r'[$R_{\mathrm{E}}$]')
ax3.set_xlim(5, -1)
ax3.set_ylim(-1, 5)
ax3.set_title('(Y, Z)')

ax4 = fig.add_subplot(224)
ax4.plot(THA_rmlatmlt_L_x, THA_rmlatmlt_L_y, color='magenta', label='THEMIS-A', linewidth=3)
ax4.scatter(THA_rmlatmlt_L_x[0],  THA_rmlatmlt_L_y[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax4.scatter(THA_rmlatmlt_L_x[-1], THA_rmlatmlt_L_y[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax4.plot(Arase_rmlatmlt_L_x, Arase_rmlatmlt_L_y, color='green', label='Arase', linewidth=3)
ax4.scatter(Arase_rmlatmlt_L_x[0],  Arase_rmlatmlt_L_y[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax4.scatter(Arase_rmlatmlt_L_x[-1], Arase_rmlatmlt_L_y[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax4, 'X‒SM\n'+ r'[$R_{\mathrm{E}}$]', 'Y‒SM\n'+ r'[$R_{\mathrm{E}}$]')
ax4.set_xlim(1, -9)
ax4.set_ylim(8.5, -1.5)
ax4.set_title(r'(X, Y) at MLAT = $0^{\circ}$')

for ax in (ax1, ax2, ax3, ax4):
    th = np.linspace(0, 2*np.pi, 256)
    ax.plot(np.cos(th), np.sin(th), 'k-', lw=1, alpha=1)
    if ax == ax1 or ax == ax2 or ax == ax4:
        # 円の中のx<0の部分を塗りつぶす
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)<0), color='k')
    if ax == ax3:
        # 円の中を塗りつぶす
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)<0), color='k')
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)>0), color='k')

    if ax == ax1:
        ax.text(-0.25, 1, '(a)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax2:
        ax.text(-0.25, 1, '(b)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax3:
        ax.text(-0.25, 1, '(c)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax4:
        ax.text(-0.25, 1, '(d)', transform=ax.transAxes, verticalalignment='top')

fig.tight_layout()
plt.show()

#fig.savefig(r"/mnt/j/KAW_observation/Figure_1_a.pdf")

# 電子・proton omniflux, PA distribution

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import pytplot as pt
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr

pt.del_data('*')

trange = ['2022-09-01/22:25', '2022-09-01/23:15']

ergpy.lepe(trange=trange, level='l2', datatype='3dflux')
ergpy.lepe(trange=trange, level='l2', datatype='omniflux')
ergpy.lepi(trange=trange, level='l2', datatype='3dflux')
ergpy.lepi(trange=trange, level='l2', datatype='omniflux')
ergpy.mgf(trange=trange, level='l2', datatype='64hz', coord='dsi')
ergpy.orb(trange=trange, level='l2')

In [ ]:
background_time_sec = 100 #[sec]

In [ ]:
import xarray as xr
import numpy as np

time_range_T    = [trange[0].replace('/', 'T'), trange[1].replace('/', 'T')]

B64_data_dsi    = psp.get_data('erg_mgf_l2_mag_64hz_dsi', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data_dsi_quality_flag   = psp.get_data('erg_mgf_l2_quality_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

qf_B, B64 = xr.align(B64_data_dsi_quality_flag[:, 3], B64_data_dsi, join='inner')

B64_data_dsi_qf = xr.where(qf_B <= 21, B64, np.nan)

ds_B64_dsi  = xr.Dataset({
    'B64_dsi_x':    B64_data_dsi_qf[:, 0],
    'B64_dsi_y':    B64_data_dsi_qf[:, 1],
    'B64_dsi_z':    B64_data_dsi_qf[:, 2]
})

ds_B64_dsi  = ds_B64_dsi.dropna(dim='time', how='all')

In [ ]:
da_mgf_spin_phase_deg           = psp.get_data('erg_mgf_l2_spin_phase_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
da_mgf_spin_phase_deg_interp    = da_mgf_spin_phase_deg.interp(time=ds_B64_dsi.time)
da_mgf_spin_phase_rad_interp    = np.deg2rad(da_mgf_spin_phase_deg_interp)

In [ ]:
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.erg_mgf_spintone_rm as emsr
importlib.reload(emsr)
os.chdir('./KAW_observation')
print(os.getcwd())

B_clean_ndarray, B_spt_ndarray, params  = emsr.remove_spintone_3comp(
    time=ds_B64_dsi.time.values,
    Bx=ds_B64_dsi['B64_dsi_x'].values,
    By=ds_B64_dsi['B64_dsi_y'].values,
    Bz=ds_B64_dsi['B64_dsi_z'].values,
    phase_rad=da_mgf_spin_phase_rad_interp.values,
    min_points=64.*3./2.
)

ds_B64_dsi_spt      = xr.Dataset({
    'B64_dsi_x_spt':    ('time', B_spt_ndarray[:, 0]),
    'B64_dsi_y_spt':    ('time', B_spt_ndarray[:, 1]),
    'B64_dsi_z_spt':    ('time', B_spt_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi.time.values})

ds_B64_dsi_clean    = xr.Dataset({
    'B64_dsi_x_clean':    ('time', B_clean_ndarray[:, 0]),
    'B64_dsi_y_clean':    ('time', B_clean_ndarray[:, 1]),
    'B64_dsi_z_clean':    ('time', B_clean_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi.time.values})

print(ds_B64_dsi_spt)
print(ds_B64_dsi_clean)

In [ ]:
time_width_B64          = (ds_B64_dsi_clean.time[10] - ds_B64_dsi_clean.time[9]) / np.timedelta64(1, 's')
ds_B_background         = ds_B64_dsi_clean.rolling(time=int(background_time_sec / time_width_B64), center=True).mean('time')

da_B_background         = ds_B_background.to_dataarray(dim='v_dim').T.dropna(dim='time', how='any')

print(da_B_background)

In [ ]:
psp.store_data('erg_mgf_l2_mag_64hz_background_dsi', data={'x': da_B_background.time, 'y': da_B_background.data})

In [ ]:
psp.projects.erg.erg_lep_part_products(
    'erg_lepe_l2_3dflux_FEDU',
    outputs=['pa'],
    pitch=[0, 180],
    energy=[40, 5000],
    mag_name='erg_mgf_l2_mag_64hz_background_dsi',
    pos_name='erg_orb_l2_pos_sm'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FPDU',
    outputs=['pa'],
    pitch=[0, 180],
    energy=[40, 1000],
    mag_name='erg_mgf_l2_mag_64hz_background_dsi',
    pos_name='erg_orb_l2_pos_sm'
)

In [ ]:
# LEP-e L3 pitch-angle distribution
ergpy.lepe(
    trange=trange,
    level="l3",
    datatype="pa",
    only_fedu=True,
)

dq_l3 = psp.get_data(
    "erg_lepe_l3_pa_FEDU",
    xarray=True,
).sel(time=slice(*trange))

# 中心エネルギーが40–5000 eVに入るchannelを選択
energy_mask = (dq_l3.v1 >= 40.0) & (dq_l3.v1 <= 5000.0)
dq_l3_band = dq_l3.where(energy_mask, drop=True)

# 機器の角度channel幅22.5 degに合わせ、隣接する11.25 deg PA binを
# 各時刻・各energy channelで非重複ペア平均する。
# 片方が欠損したペアを有効値にしないため、PA平均ではskipna=Falseとする。
pa_coord = dq_l3_band["spec_bins"]
if pa_coord.ndim != 1:
    raise ValueError(f"Expected 1-D L3 PA coordinate, got dims={pa_coord.dims}")

pa_dim = pa_coord.dims[0]
n_pa = dq_l3_band.sizes[pa_dim]
if n_pa % 2 != 0:
    raise ValueError(f"PA bin数が偶数でないため2-bin rebinできない: {n_pa}")

pa_values = np.asarray(pa_coord.values, dtype=float)
if not np.allclose(np.diff(pa_values), 11.25, rtol=0.0, atol=1e-6):
    raise ValueError(f"Expected 11.25-deg L3 PA spacing, got {pa_values}")

dq_l3_band_22p5 = dq_l3_band.coarsen(
    {pa_dim: 2},
    boundary="exact",
).mean(
    skipna=False,
    keep_attrs=True,
)

# PAを22.5 degへrebinした後、energy channel間を算術平均する。
dq_electron_pa = dq_l3_band_22p5.mean(
    dim="v1_dim",
    skipna=True,
    keep_attrs=True,
)
dq_electron_pa.attrs["pitch_angle_rebin"] = (
    "Adjacent 11.25-deg bins averaged into non-overlapping 22.5-deg bins; "
    "both bins required at each time and energy"
)

print("LEP-e L3 PA centers after 22.5-deg rebin:", dq_electron_pa.spec_bins.values)

In [ ]:
mpl.rcParams['font.size'] = 20

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import pytplot as pt

def omniflux_plot_Arase(inst_name, fig, ax, cax, dq, vmin=None, vmax=None, energy_min=None, energy_max=None, ytitle=None, ztitle=None):
    # 取り出し（全て (T, E) 形状）
    flux = np.asarray(dq.data, dtype=float)           # (T, E)
    if inst_name == 'lepe':
        E    = np.asarray(dq.spec_bins, dtype=float)      # (T, E) 変動エネルギー
    elif inst_name == 'lepi':
        E    = np.asarray(dq.spec_bins, dtype=float)      # (E) 変動エネルギー
        E    = E * 1E3                                     # keV → eV
        E    = np.broadcast_to(E[None, :], flux.shape)      # (T, E)
    t    = np.asarray(dq.time.values)                 # (T,)

    print("before:", flux.shape, E.shape)

    # 列方向で有限なエネルギーチャンネルだけ残す
    if E.ndim == 2:
        good_e = np.all(np.isfinite(E), axis=0)
    elif E.ndim == 1:
        good_e = np.isfinite(E)
    else:
        raise ValueError(f"unexpected E.ndim={E.ndim}, E.shape={E.shape}")

    if not np.any(good_e):
        raise ValueError("有効なエネルギーチャンネルがありません。")

    flux = flux[:, good_e]
    if E.ndim == 2:
        E = E[:, good_e]
    else:
        E = np.broadcast_to(E[None, good_e], flux.shape)

    print("after :", flux.shape, E.shape)

    # LogNorm 用
    flux = flux.copy()
    flux[~np.isfinite(flux)] = np.nan
    flux[flux <= 0] = np.nan

    if np.all(~np.isfinite(flux)):
        raise ValueError("有効な（>0）フラックスがありません。")

    if vmin == None:
        vmin = np.nanmin(flux)
    if vmax == None:
        vmax = np.nanmax(flux)
    if not (vmin < vmax):
        vmax = vmin * 1.0001
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    print(vmin, vmax)

    flux[np.isnan(flux)] = 1E-99

    # ---- 時間メッシュ生成（(T,E)）----
    Tmesh = np.broadcast_to(t[:, None], E.shape)
    print(Tmesh.shape)

    # ---- 描画 ----
    mesh = ax.pcolormesh(Tmesh, E, flux, norm=norm, cmap=cm.turbo, shading='nearest')
    cb = fig.colorbar(mesh, cax=cax)
    cb.set_label("")  # ラベル消す
    #if ztitle:
    #    ax.text(0.85, 0.15, ztitle,
    #            transform=ax.transAxes, ha='center', va='center', color='k',
    #            bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.25'))

    ax.set_yscale('log')
    ax.set_ylabel(ytitle)
    ax.grid(which='both', alpha=0.5)
    ax.minorticks_on()

    if energy_min != None:
        ax.set_ylim(ymin=energy_min)
    if energy_max != None:
        ax.set_ylim(ymax=energy_max)

    # 時刻目盛り
    loc = mdates.AutoDateLocator()
    fmt = mdates.ConciseDateFormatter(loc)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(fmt)

    return fig, ax, cax

def pa_plot_Arase(fig, ax, cax, dq, vmin=None, vmax=None, ytitle=None, ztitle=None):
    # 取り出し（全て (T, E) 形状）
    flux = np.asarray(dq.data, dtype=float)           # (T, E)
    PA    = np.asarray(dq.spec_bins, dtype=float)     # (T, E)
    t    = np.asarray(dq.time.values)                 # (T,)

    if PA.ndim == 1:
        PA = np.broadcast_to(PA[None, :], flux.shape)

    # pcolormeshのX/Yは非有限NG → 列方向で全部有限なチャンネルだけ残す
    good_PA = np.all(~np.isnan(PA), axis=0)
    if not np.all(good_PA):
        flux = flux[:, good_PA]
        PA    = PA[:,    good_PA]

    # ---- 値の前処理（LogNorm 用）----
    flux[~np.isfinite(flux)] = np.nan
    flux[flux <= 0] = np.nan
    if np.all(~np.isfinite(flux)):
        raise ValueError("有効な（>0）フラックスがありません。")

    if vmin == None:
        vmin = np.nanmin(flux)
    if vmax == None:
        vmax = np.nanmax(flux)
    if not (vmin < vmax):
        vmax = vmin * 1.0001
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    print(vmin, vmax)

    flux[np.isnan(flux)] = 1E-99

    # ---- 時間メッシュ生成（(T,E)）----
    Tmesh = np.broadcast_to(t[:, None], PA.shape)

    print(flux.shape)
    print(PA.shape)
    print(Tmesh.shape)

    # ---- 描画 ----
    mesh = ax.pcolormesh(Tmesh, PA, flux, norm=norm, cmap=cm.turbo, shading='nearest')
    cb = fig.colorbar(mesh, cax=cax)
    cb.set_label("")  # ラベル消す
    #if ztitle:
    #    ax.text(0.85, 0.5, ztitle,
    #            transform=ax.transAxes, ha='center', va='center', color='k',
    #            bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.25'))

    ax.set_ylabel(ytitle)
    ax.grid(which='both', alpha=0.5)
    ax.minorticks_on()

    ax.set_ylim(ymax=180, ymin=0)
    ax.set_yticks(np.arange(0, 181, 45))

    # 時刻目盛り
    loc = mdates.AutoDateLocator()
    fmt = mdates.ConciseDateFormatter(loc)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(fmt)

    return fig, ax, cax

def add_panel_label(ax, label, x=-0.08, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

fig = plt.figure(figsize=(10, 12))
gs = fig.add_gridspec(4, 2, width_ratios=[1, 0.025], wspace=0.05)
ax_1 = fig.add_subplot(gs[0, 0])
cax_1 = fig.add_subplot(gs[0, 1])
ax_2 = fig.add_subplot(gs[1, 0], sharex=ax_1)
cax_2 = fig.add_subplot(gs[1, 1])
ax_3 = fig.add_subplot(gs[2, 0], sharex=ax_1)
cax_3 = fig.add_subplot(gs[2, 1])
ax_4 = fig.add_subplot(gs[3, 0], sharex=ax_1)
cax_4 = fig.add_subplot(gs[3, 1])

ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)
ax_3.tick_params(axis='x', which='both', labelbottom=False)

dq_proton = psp.get_data('erg_lepi_l2_omniflux_FPDO', xarray=True).sel(time=slice(*trange)) * 1E-3 # '#/s/$cm^{2}$/str/eV'
dq_proton_pa = psp.get_data('erg_lepi_l2_3dflux_FPDU_pa', xarray=True).sel(time=slice(*trange))    #'#/s/$cm^{2}$/str/eV'

dq_electron = psp.get_data('erg_lepe_l2_omniflux_FEDO', xarray=True).sel(time=slice(*trange))      #'#/s/$cm^{2}$/str/eV'
#dq_electron_pa = psp.get_data('erg_lepe_l2_3dflux_FEDU_pa', xarray=True).sel(time=slice(*trange))  #'#/s/$cm^{2}$/str/eV'

ytitle_proton = r'LEP-i $\mathrm{H}^{+}$' + '\n' + r'[eV/q]'
ztitle_proton = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_Arase('lepi', fig, ax_1, cax_1, dq_proton, 1E0, 1E4, 4E1, 3E4, ytitle_proton, ztitle_proton)

ytitle_proton_pa = r'LEP-i $\mathrm{H}^{+}$' + '\n' + r'[deg]'
ztitle_proton_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_Arase(fig, ax_2, cax_2, dq_proton_pa, 1E0, 1E4, ytitle_proton_pa, ztitle_proton_pa)

ytitle_electron = r'LEP-e $\mathrm{e}^{-}$' + '\n' + r'[eV]'
ztitle_electron = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_Arase('lepe', fig, ax_3, cax_3, dq_electron, 1E2, 1E5, None, None, ytitle_electron, ztitle_electron)

ytitle_electron_pa = r'LEP-e $\mathrm{e}^{-}$' + '\n' + r'[deg]'
ztitle_electron_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_Arase(fig, ax_4, cax_4, dq_electron_pa, 1E2, 1E5, ytitle_electron_pa, ztitle_electron_pa)

add_panel_label(ax_1, '(b-1)')
add_panel_label(ax_2, '(b-2)')
add_panel_label(ax_3, '(b-3)')
add_panel_label(ax_4, '(b-4)')

plt.tight_layout()
plt.show()

In [ ]:
#psp.projects.themis.esa(trange=trange, probe='a', level='l2', no_update=True)
#
#themis_a_e_omniflux = psp.get_data('tha_peef_en_eflux', xarray=True)
#themis_a_e_omniflux = themis_a_e_omniflux.sel(time=slice(trange[0], trange[1]))
#print(themis_a_e_omniflux)
#
#themis_a_proton_omniflux = psp.get_data('tha_peif_en_eflux', xarray=True)
#themis_a_proton_omniflux = themis_a_proton_omniflux.sel(time=slice(trange[0], trange[1]))
#print(themis_a_proton_omniflux)

In [ ]:
#import os
#
#import sys, importlib
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.themis_band_dnumflux as tbd
#importlib.reload(tbd)
#
#tbd.compute_band_dnumflux(
#    folder=r'/mnt/j/observation_data/themis/tha/idl_output/per_bin',
#    spec='peif',
#    Emin=40,
#    Emax=1000
#)
#
#tbd.compute_band_dnumflux(
#    folder=r'/mnt/j/observation_data/themis/tha/idl_output/per_bin',
#    spec='peef',
#    Emin=40,
#    Emax=5000
#)

In [ ]:
#import pandas as pd, numpy as np, xarray as xr
#
#def csv_to_xarray_dnumflux(csv_path: str, var_name: str = "dnumflux"):
#    df = pd.read_csv(csv_path)
#    if "time_iso" not in df.columns:
#        raise ValueError("CSVに'time_iso'列が必要です。")
#    # ピッチ角中心（列名）→ spec_bins 座標
#    pa_cols = [c for c in df.columns if c != "time_iso"]
#    pa_vals = np.array([float(c) for c in pa_cols])  # 度
#
#    # 時間
#    time = pd.to_datetime(df["time_iso"], format="%Y-%m-%d/%H:%M:%S", errors="coerce").to_numpy()
#
#    # データ (len(time), len(pitch_angle))
#    data = df[pa_cols].to_numpy(dtype=float)
#
#    da = xr.DataArray(
#        data,
#        dims=("time", "spec_bins"),
#        coords={"time": time, "spec_bins": pa_vals},
#        name=var_name,
#        attrs={
#            "units": "1/(cm^2 sr s eV)",
#            "long_name": "band-averaged differential number flux",
#        }
#    )
#    return da
#
#da_tha_proton_pa = csv_to_xarray_dnumflux(
#    r"/mnt/j/observation_data/themis/tha/idl_output/per_bin/tha_peif_dnumflux_E40to1000.csv",
#    var_name="tha_peif_dnumflux_E40to1000"
#    )
#
#da_tha_electron_pa = csv_to_xarray_dnumflux(
#    r"/mnt/j/observation_data/themis/tha/idl_output/per_bin/tha_peef_dnumflux_E40to5000.csv",
#    var_name="tha_peef_dnumflux_E40to5000"
#    )
#
#print(da_tha_proton_pa)
#print(da_tha_electron_pa)

In [ ]:
dn_tha_proton = xr.open_dataset('/mnt/j/observation_data/themis/tha/idl_output/omni_flux/tha_peir_omni_diff_number_flux.nc')
dn_tha_electron = xr.open_dataset('/mnt/j/observation_data/themis/tha/idl_output/omni_flux/tha_peer_omni_diff_number_flux.nc')

In [ ]:
da_tha_proton_pa = xr.open_dataset('/mnt/j/observation_data/themis/tha/idl_output/tha_peir_flux_pa_40_1000eV_20220901_2225_2315.nc')
da_tha_electron_pa = xr.open_dataset('/mnt/j/observation_data/themis/tha/idl_output/tha_peef_flux_pa_40_5000eV_20220901_2225_2315.nc')

In [ ]:
print(da_tha_proton_pa)

In [ ]:
#da_tha_proton = psp.get_data('tha_peif_en_eflux', xarray=True)
#da_tha_electron = psp.get_data('tha_peef_en_eflux', xarray=True)
#
#print(da_tha_proton)
#print(da_tha_electron)
#
#import os
#import sys
#import importlib
#from pathlib import Path
#
#ROOT = Path("/home/satanka/Documents/observation_workspace")
#if str(ROOT) not in sys.path:
#    sys.path.insert(0, str(ROOT))
#
#import module_handmade.xarray_numflux_utils as xnu
#importlib.reload(xnu)
#
#dn_tha_proton = xnu.eflux_to_numflux(da_eflux=da_tha_proton)
#dn_tha_electron = xnu.eflux_to_numflux(da_eflux=da_tha_electron)
#
#print(dn_tha_proton)
#print(dn_tha_electron)

In [ ]:
dn_tha_proton

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import pytplot as pt

def omniflux_plot_THEMIS_A(fig, ax, cax, dq, vmin=None, vmax=None, energy_min=None, energy_max=None, ytitle=None, ztitle=None):
    # 取り出し（全て (T, E) 形状）
    flux = np.asarray(dq.diff_number_flux, dtype=float)           # (T, E)
    E    = np.asarray(dq.energy, dtype=float)      # (T, E) 変動エネルギー
    t    = np.asarray(dq.time.values)                 # (T,)

    # pcolormeshのX/Yは非有限NG → 列方向で全部有限なチャンネルだけ残す
    print("before:", flux.shape, E.shape)

    # 列方向で有限なエネルギーチャンネルだけ残す
    if E.ndim == 2:
        good_e = np.all(np.isfinite(E), axis=0)
    elif E.ndim == 1:
        good_e = np.isfinite(E)
    else:
        raise ValueError(f"unexpected E.ndim={E.ndim}, E.shape={E.shape}")

    if not np.any(good_e):
        raise ValueError("有効なエネルギーチャンネルがありません。")

    flux = flux[:, good_e]
    if E.ndim == 2:
        E = E[:, good_e]
    else:
        E = np.broadcast_to(E[None, good_e], flux.shape)

    print("after :", flux.shape, E.shape)

    # ---- 値の前処理（LogNorm 用）----
    flux[~np.isfinite(flux)] = np.nan
    flux[flux <= 0] = np.nan
    if np.all(~np.isfinite(flux)):
        raise ValueError("有効な（>0）フラックスがありません。")

    if vmin == None:
        vmin = np.nanmin(flux)
    if vmax == None:
        vmax = np.nanmax(flux)
    if not (vmin < vmax):
        vmax = vmin * 1.0001
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    #print(vmin, vmax)

    flux[np.isnan(flux)] = 1E-99

    # ---- 時間メッシュ生成（(T,E)）----
    Tmesh = np.broadcast_to(t[:, None], E.shape)

    #print(flux.shape)
    #print(E.shape)
    #print(Tmesh.shape)

    # ---- 描画 ----
    mesh = ax.pcolormesh(Tmesh, E, flux, norm=norm, cmap=cm.turbo, shading='nearest')
    cb = fig.colorbar(mesh, cax=cax)
    cb.set_label("")  # ラベル消す
    #if ztitle:
    #    ax.text(0.85, 0.15, ztitle,
    #            transform=ax.transAxes, ha='center', va='center', color='k',
    #            bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.25'))

    ax.set_yscale('log')
    ax.set_ylabel(ytitle)
    ax.grid(which='both', alpha=0.5)
    ax.minorticks_on()

    if energy_min != None:
        ax.set_ylim(ymin=energy_min)
    if energy_max != None:
        ax.set_ylim(ymax=energy_max)

    # 時刻目盛り
    loc = mdates.AutoDateLocator()
    fmt = mdates.ConciseDateFormatter(loc)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(fmt)

    return fig, ax, cax

def pa_plot_THEMIS_A(fig, ax, cax, dq, vmin=None, vmax=None, ytitle=None, ztitle=None):
    # 取り出し（全て (T, E) 形状）
    flux = np.asarray(dq.diff_number_flux, dtype=float)           # (T, E)
    PA    = np.asarray(dq.pitch_angle, dtype=float)     # (E)
    PA    = np.broadcast_to(PA[None, :], flux.shape)  # (T, E)
    t    = np.asarray(dq.time.values)                 # (T,)
    print(flux)

    # pcolormeshのX/Yは非有限NG → 列方向で全部有限なチャンネルだけ残す
    good_PA = np.all(~np.isnan(PA), axis=0)
    if not np.all(good_PA):
        flux = flux[:, good_PA]
        PA    = PA[:,    good_PA]

    # ---- 値の前処理（LogNorm 用）----
    flux[~np.isfinite(flux)] = np.nan
    flux[flux <= 0] = np.nan
    if np.all(~np.isfinite(flux)):
        raise ValueError("有効な（>0）フラックスがありません。")

    if vmin == None:
        vmin = np.nanmin(flux)
    if vmax == None:
        vmax = np.nanmax(flux)
    if not (vmin < vmax):
        vmax = vmin * 1.0001
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    #print(vmin, vmax)

    flux[np.isnan(flux)] = 1E-99

    # ---- 時間メッシュ生成（(T,E)）----
    print(t.shape)
    if t.shape == flux.shape:
        Tmesh = t
    else:
        Tmesh = np.broadcast_to(t[:, None], flux.shape)

    #print(flux.shape)
    #print(PA.shape)
    #print(Tmesh.shape)

    # ---- 描画 ----
    mesh = ax.pcolormesh(Tmesh, PA, flux, norm=norm, cmap=cm.turbo, shading='nearest')
    cb = fig.colorbar(mesh, cax=cax)
    cb.set_label("")  # ラベル消す
    #if ztitle:
    #    ax.text(0.85, 0.5, ztitle,
    #            transform=ax.transAxes, ha='center', va='center', color='k',
    #            bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.25'))

    ax.set_ylabel(ytitle)
    ax.grid(which='both', alpha=0.5)
    ax.minorticks_on()

    ax.set_ylim(ymax=180, ymin=0)
    ax.set_yticks(np.arange(0, 181, 45))

    # 時刻目盛り
    loc = mdates.AutoDateLocator()
    fmt = mdates.ConciseDateFormatter(loc)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(fmt)

    return fig, ax, cax


fig = plt.figure(figsize=(10, 12))
gs = fig.add_gridspec(4, 2, width_ratios=[1, 0.025], wspace=0.05)
ax_1 = fig.add_subplot(gs[0, 0])
cax_1 = fig.add_subplot(gs[0, 1])
ax_2 = fig.add_subplot(gs[1, 0], sharex=ax_1)
cax_2 = fig.add_subplot(gs[1, 1])
ax_3 = fig.add_subplot(gs[2, 0], sharex=ax_1)
cax_3 = fig.add_subplot(gs[2, 1])
ax_4 = fig.add_subplot(gs[3, 0], sharex=ax_1)
cax_4 = fig.add_subplot(gs[3, 1])

ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)
ax_3.tick_params(axis='x', which='both', labelbottom=False)

dn_tha_proton = dn_tha_proton.sel(time=slice(*trange))
dn_tha_electron = dn_tha_electron.sel(time=slice(*trange))
da_tha_proton_pa = da_tha_proton_pa.sel(time=slice(*trange))
da_tha_electron_pa = da_tha_electron_pa.sel(time=slice(*trange))

ytitle_proton = r'ESA ion' + '\n' + r'[eV/q]'
ztitle_proton = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_THEMIS_A(fig, ax_1, cax_1, dn_tha_proton, 1E0, 1E4, None, None, ytitle_proton, ztitle_proton)

ytitle_proton_pa = r'ESA ion' + '\n' + r'[deg]'
ztitle_proton_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_THEMIS_A(fig, ax_2, cax_2, da_tha_proton_pa, 1E0, 1E4, ytitle_proton_pa, ztitle_proton_pa)

ytitle_electron = r'ESA $\mathrm{e}^{-}$' + '\n' + r'[eV]'
ztitle_electron = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_THEMIS_A(fig, ax_3, cax_3, dn_tha_electron, 1E2, 1E5, None, None, ytitle_electron, ztitle_electron)

ytitle_electron_pa = r'ESA $\mathrm{e}^{-}$' + '\n' + r'[deg]'
ztitle_electron_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_THEMIS_A(fig, ax_4, cax_4, da_tha_electron_pa, 1E2, 1E5, ytitle_electron_pa, ztitle_electron_pa)

add_panel_label(ax_1, '(c-1)')
add_panel_label(ax_2, '(c-2)')
add_panel_label(ax_3, '(c-3)')
add_panel_label(ax_4, '(c-4)')

plt.tight_layout()
plt.show()


# $|\bf{E}_{\perp}|$、$|\bf{B}_{\perp}|$、$S_{\parallel}$のplot

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import pytplot as pt

time_range = ['20220901/21:00:00', '20220902/00:00:00']

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_Arase/2022-09-01/2230-2330'

ergpy.pwe_efd(trange=time_range, level='l2', datatype='64', coord='dsi', get_support_data=True)
ergpy.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi', get_support_data=True)

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]

B64_data_dsi    = psp.get_data('erg_mgf_l2_mag_64hz_dsi', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data_dsi_quality_flag   = psp.get_data('erg_mgf_l2_quality_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

qf_B, B64 = xr.align(B64_data_dsi_quality_flag[:, 3], B64_data_dsi, join='inner')

B64_data_dsi_qf = xr.where(qf_B <= 21, B64, np.nan)

ds_B64_dsi  = xr.Dataset({
    'B64_dsi_x':    B64_data_dsi_qf[:, 0],
    'B64_dsi_y':    B64_data_dsi_qf[:, 1],
    'B64_dsi_z':    B64_data_dsi_qf[:, 2]
})

ds_B64_dsi  = ds_B64_dsi.dropna(dim='time', how='all')

def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

ds_B64_dsi_segs = split_by_gap(ds_B64_dsi, gap_thr=np.timedelta64(63, 'ms'))

ds_B64_dsi_seg0 = ds_B64_dsi_segs[0]

da_mgf_spin_phase_deg           = psp.get_data('erg_mgf_l2_spin_phase_64hz', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
da_mgf_spin_phase_deg_interp    = da_mgf_spin_phase_deg.interp(time=ds_B64_dsi.time)
da_mgf_spin_phase_rad_interp    = np.deg2rad(da_mgf_spin_phase_deg_interp)

import os
import sys
import importlib
from pathlib import Path

ROOT = Path("/home/satanka/Documents/observation_workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import module_handmade.erg_mgf_spintone_rm as emsr
importlib.reload(emsr)

B_clean_ndarray, B_spt_ndarray, params  = emsr.remove_spintone_3comp(
    time=ds_B64_dsi_seg0.time.values,
    Bx=ds_B64_dsi_seg0['B64_dsi_x'].values,
    By=ds_B64_dsi_seg0['B64_dsi_y'].values,
    Bz=ds_B64_dsi_seg0['B64_dsi_z'].values,
    phase_rad=da_mgf_spin_phase_rad_interp.values,
    min_points=64.*3./2.
)

ds_B64_dsi_seg0_spt      = xr.Dataset({
    'B64_dsi_x_spt':    ('time', B_spt_ndarray[:, 0]),
    'B64_dsi_y_spt':    ('time', B_spt_ndarray[:, 1]),
    'B64_dsi_z_spt':    ('time', B_spt_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi_seg0.time.values})

ds_B64_dsi_seg0_clean    = xr.Dataset({
    'B64_dsi_x_clean':    ('time', B_clean_ndarray[:, 0]),
    'B64_dsi_y_clean':    ('time', B_clean_ndarray[:, 1]),
    'B64_dsi_z_clean':    ('time', B_clean_ndarray[:, 2])
    }, coords={'time': ds_B64_dsi_seg0.time.values})

E64_data_dsi_x  = psp.get_data('erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_y  = psp.get_data('erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_dsi_quality_flag = psp.get_data('erg_pwe_efd_l2_E64Hz_dsi_quality_flag', xarray=True).sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

# QF = 0 の時間を抽出
bad_times = E64_data_dsi_quality_flag.time.where(E64_data_dsi_quality_flag != 0, drop=True)

E64_data_dsi_x_qf = E64_data_dsi_x.where(E64_data_dsi_quality_flag.interp(time=E64_data_dsi_x.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)
E64_data_dsi_y_qf = E64_data_dsi_y.where(E64_data_dsi_quality_flag.interp(time=E64_data_dsi_y.time, method='nearest', kwargs={'fill_value': 'extrapolate'}) == 0, np.nan)

ds_E64_dsi_xy   = xr.Dataset({
    'E64_dsi_x': E64_data_dsi_x_qf,
    'E64_dsi_y': E64_data_dsi_y_qf
})
ds_E64_dsi_xy   = ds_E64_dsi_xy.dropna(dim='time', how='all')

ds_E64_dsi_xy_segs = split_by_gap(ds_E64_dsi_xy, gap_thr=np.timedelta64(63, 'ms'))

def make_ds_EB_func(ds_E_xy, E_vars, ds_B_xyz, B_vars, output_vars):
    time_base   = ds_E_xy.time
    ds_B_xyz_interp  = ds_B_xyz.interp(time=time_base, method='linear')

    da_Ex   = ds_E_xy[E_vars[0]]
    da_Ey   = ds_E_xy[E_vars[1]]
    da_Bx   = ds_B_xyz_interp[B_vars[0]]
    da_By   = ds_B_xyz_interp[B_vars[1]]
    da_Bz   = ds_B_xyz_interp[B_vars[2]]

    da_Ez   = xr.where(np.abs(da_Bz) > 1E-2, -(da_Ex * da_Bx + da_Ey * da_By) / da_Bz, np.nan)

    ds_EB   = xr.Dataset({
        output_vars[0]: da_Ex,
        output_vars[1]: da_Ey,
        output_vars[2]: da_Ez,
        output_vars[3]: da_Bx,
        output_vars[4]: da_By,
        output_vars[5]: da_Bz,
    })

    ds_EB   = ds_EB.dropna(dim='time', how='any')

    return ds_EB

E64_xy_vars     = ['E64_dsi_x', 'E64_dsi_y']
B64_xyz_vars    = ['B64_dsi_x_clean', 'B64_dsi_y_clean', 'B64_dsi_z_clean']
EB64_vars       = ['E64_dsi_x', 'E64_dsi_y', 'E64_dsi_z', 'B64_dsi_x', 'B64_dsi_y', 'B64_dsi_z']

ds_EB64_dsi_segs    = []

for count in range(len(ds_E64_dsi_xy_segs)):
    ds_     = make_ds_EB_func(ds_E64_dsi_xy_segs[count], E64_xy_vars, ds_B64_dsi_seg0_clean, B64_xyz_vars, EB64_vars)
    ds_EB64_dsi_segs.append(ds_)

ergpy.orb(trange=time_range, level='l2', datatype='def', no_update=True)
da_Arase_pos_gsm    = psp.get_data('erg_orb_l2_pos_gsm', xarray=True)
da_Arase_pos_gsm    = da_Arase_pos_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

da_Arase_pos_unit_gsm   = da_Arase_pos_gsm / np.sqrt((da_Arase_pos_gsm * da_Arase_pos_gsm).sum(dim='v_dim'))

psp.store_data('Arase_pos_unit_gsm', data={'x': da_Arase_pos_unit_gsm.time, 'y': da_Arase_pos_unit_gsm.data})
psp.cotrans(name_in='Arase_pos_unit_gsm', name_out='Arase_pos_unit_j2000', coord_in='gsm', coord_out='j2000')
psp.projects.erg.erg_cotrans(in_name='Arase_pos_unit_j2000', out_name='Arase_pos_unit_dsi', in_coord='j2000', out_coord='dsi')

da_Arase_pos_unit_dsi   = psp.get_data('Arase_pos_unit_dsi', xarray=True)

background_time_sec = 100 #[sec]

time_width_B64          = (ds_B64_dsi_seg0_clean.time[10] - ds_B64_dsi_seg0_clean.time[9]) / np.timedelta64(1, 's')
ds_B_background         = ds_B64_dsi_seg0_clean.rolling(time=int(background_time_sec / time_width_B64), center=True).mean('time')

da_B_background         = ds_B_background.to_dataarray(dim='v_dim').T.dropna(dim='time', how='any')

print(da_B_background)

da_B_background_unit    = da_B_background / np.sqrt((da_B_background * da_B_background).sum(dim='v_dim'))
da_B_background_unit    = da_B_background_unit.dropna(how='all', dim='time')

print(da_B_background_unit)
print(np.nanmin(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim'))))
print(np.nanmax(np.sqrt((da_B_background_unit * da_B_background_unit).sum(dim='v_dim'))))

da_Arase_pos_unit_dsi_interp    = da_Arase_pos_unit_dsi.interp(time=da_B_background_unit.time)

da_u_   = da_Arase_pos_unit_dsi_interp - (da_Arase_pos_unit_dsi_interp * da_B_background_unit).sum(dim='v_dim') * da_B_background_unit

da_e_z_FAC_inDSI    = da_B_background_unit.drop_attrs()
da_e_x_FAC_inDSI    = (da_u_ / np.sqrt((da_u_ * da_u_).sum(dim='v_dim'))).drop_attrs()
da_e_y_FAC_inDSI    = (xr.apply_ufunc(np.cross, da_e_z_FAC_inDSI, da_e_x_FAC_inDSI, input_core_dims=[['v_dim'], ['v_dim']], output_core_dims=[['v_dim']], vectorize=True)).drop_attrs()

R_FAC_to_DSI = xr.concat(
    [da_e_x_FAC_inDSI, da_e_y_FAC_inDSI, da_e_z_FAC_inDSI],
    dim='axis'
)
R_FAC_to_DSI = R_FAC_to_DSI.assign_coords(axis=['x_FAC', 'y_FAC', 'z_FAC']).assign_coords(v_dim=np.arange(3))

R_DSI_to_FAC = R_FAC_to_DSI.transpose('time', 'v_dim', 'axis')

da_e_x_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=0)
da_e_y_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=1)
da_e_z_DSI_inFAC = R_DSI_to_FAC.sel(v_dim=2)

ds_EB64_fac_segs = []

for ds_seg in ds_EB64_dsi_segs:
    # 1) この seg の時間に合わせて回転行列を補間
    R_seg = R_DSI_to_FAC.interp(time=ds_seg.time)

    # 2) DSIベクトル (time, v_dim) を作る
    E_dsi = xr.concat(
        [ds_seg['E64_dsi_x'], ds_seg['E64_dsi_y'], ds_seg['E64_dsi_z']],
        dim='v_dim'
    )
    B_dsi = xr.concat(
        [ds_seg['B64_dsi_x'], ds_seg['B64_dsi_y'], ds_seg['B64_dsi_z']],
        dim='v_dim'
    )

    # こちらも v_dim を 0,1,2 にそろえる
    E_dsi = E_dsi.assign_coords(v_dim=np.arange(3))
    B_dsi = B_dsi.assign_coords(v_dim=np.arange(3))

    # 3) DSI → FAC 回転
    E_fac = xr.dot(E_dsi, R_seg, dims='v_dim')  # (time, axis)
    B_fac = xr.dot(B_dsi, R_seg, dims='v_dim')

    # 4) axis 次元を変数に落とす
    E_fac_ds = (
        E_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'E64_fac_x',
            'y_FAC': 'E64_fac_y',
            'z_FAC': 'E64_fac_z',
        })
    )

    B_fac_ds = (
        B_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'B64_fac_x',
            'y_FAC': 'B64_fac_y',
            'z_FAC': 'B64_fac_z',
        })
    )

    ds_fac = xr.merge([E_fac_ds, B_fac_ds])
    ds_fac = ds_fac.assign_attrs(ds_seg.attrs)
    ds_EB64_fac_segs.append(ds_fac)

In [ ]:
ds_EB64_fac_segs

In [ ]:
ds_fac = xr.concat((ds_EB64_fac_segs[0], ds_EB64_fac_segs[1], ds_EB64_fac_segs[2], ds_EB64_fac_segs[3], ds_EB64_fac_segs[4], ds_EB64_fac_segs[5], ds_EB64_fac_segs[6]), dim='time')
print(ds_fac.time[0])
print(ds_fac.time[-1])

In [ ]:
Eperp_2 = ((ds_fac['E64_fac_x'].data)**2E0 + (ds_fac['E64_fac_y'].data)**2E0)
Bperp_2 = ((ds_fac['B64_fac_x'].data)**2E0 + (ds_fac['B64_fac_y'].data)**2E0)

mu_0 = 4.*np.pi * 1E-7

S_para = (ds_fac['E64_fac_x'].data * ds_fac['B64_fac_y'].data - ds_fac['E64_fac_y'].data * ds_fac['B64_fac_x'].data) / mu_0 * 1E-12 #W/m^2

In [ ]:
import matplotlib.ticker as mticker
from datetime import datetime

mpl.rcParams['font.size'] = 25

fig = plt.figure(figsize=(10, 22))

gs = fig.add_gridspec(7, 2, width_ratios=[1, 0.025], wspace=0.05, hspace=0.15)
ax_1 = fig.add_subplot(gs[0, 0])
cax_1 = fig.add_subplot(gs[0, 1])
ax_2 = fig.add_subplot(gs[1, 0], sharex=ax_1)
cax_2 = fig.add_subplot(gs[1, 1])
ax_3 = fig.add_subplot(gs[2, 0], sharex=ax_1)
cax_3 = fig.add_subplot(gs[2, 1])
ax_4 = fig.add_subplot(gs[3, 0], sharex=ax_1)
cax_4 = fig.add_subplot(gs[3, 1])
ax_5 = fig.add_subplot(gs[4, 0], sharex=ax_1)
ax_6 = fig.add_subplot(gs[5, 0], sharex=ax_1)
ax_7 = fig.add_subplot(gs[6, 0], sharex=ax_1)

ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)
ax_3.tick_params(axis='x', which='both', labelbottom=False)
ax_4.tick_params(axis='x', which='both', labelbottom=False)
ax_5.tick_params(axis='x', which='both', labelbottom=False)
ax_6.tick_params(axis='x', which='both', labelbottom=False)

ytitle_proton = r'LEP-i $\mathrm{H}^{+}$' + '\n' + r'[eV/q]'
ztitle_proton = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_Arase('lepi', fig, ax_1, cax_1, dq_proton, np.nanpercentile(dq_proton, 25), np.nanpercentile(dq_proton, 99), 5E1, 1E4, ytitle_proton, ztitle_proton)

ytitle_proton_pa = r'LEP-i $\mathrm{H}^{+}$' + '\n' + r'[deg]'
ztitle_proton_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_Arase(fig, ax_2, cax_2, dq_proton_pa, np.nanpercentile(dq_proton_pa, 10), np.nanpercentile(dq_proton_pa, 99), ytitle_proton_pa, ztitle_proton_pa)

ytitle_electron = r'LEP-e $\mathrm{e}^{-}$' + '\n' + r'[eV]'
ztitle_electron = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_Arase('lepe', fig, ax_3, cax_3, dq_electron, np.nanpercentile(dq_electron, 25), np.nanpercentile(dq_electron, 99), 5E1, 1E4, ytitle_electron, ztitle_electron)

ytitle_electron_pa = r'LEP-e $\mathrm{e}^{-}$' + '\n' + r'[deg]'
ztitle_electron_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_Arase(fig, ax_4, cax_4, dq_electron_pa, np.nanpercentile(dq_electron_pa, 10), np.nanpercentile(dq_electron_pa, 99), ytitle_electron_pa, ztitle_electron_pa)

ax_5.scatter(ds_fac.time, np.sqrt(Eperp_2), c='k', s=0.1)
#ax_5.plot(ds_fac.time, np.sqrt(Eperp_2), c='k', linewidth=0.5)
ax_5.set_ylabel(r'$|\mathbf{E}_{\perp}|$' + '\n' + r'[$\mathrm{mV/m}$]')
ax_5.set_ylim(ymin=0, ymax=120)
ax_5.minorticks_on()
ax_5.grid(which='both', alpha=0.5)

ax_6.scatter(ds_fac.time, np.sqrt(Bperp_2), c='k', s=0.1)
#ax_6.plot(ds_fac.time, np.sqrt(Bperp_2), c='k', linewidth=0.5)
ax_6.set_ylabel(r'$|\mathbf{B}_{\perp}|$' + '\n' + r'[$\mathrm{nT}$]')
ax_6.set_ylim(ymin=0)
ax_6.minorticks_on()
ax_6.grid(which='both', alpha=0.5)

ax_7.scatter(ds_fac.time, S_para*1E3, c='k', s=0.1)
#ax_7.plot(ds_fac.time, S_para*1E3, c='k', linewidth=0.5)
ax_7.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
ax_7.set_ylim(ymin=-0.15, ymax=0.5)
#ax_7.set_yscale('symlog')
ax_7.minorticks_on()
ax_7.grid(which='both', alpha=0.5)


time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_7.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
ax_7.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax_7.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))

def add_panel_label(ax, label, x=-0.15, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

def to_py_datetime(t_np64):
    return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)

# 軌道データ
pos_da = psp.get_data('erg_orb_l2_pos_gsm', xarray=True)  # (Nt, 3)
t_pos_py = to_py_datetime(pos_da.time.values)
t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数

pos_X   = np.asarray(pos_da[:, 0], dtype=float)
pos_Y   = np.asarray(pos_da[:, 1], dtype=float)
pos_Z   = np.asarray(pos_da[:, 2], dtype=float)

# 補間関数（tick の x は「日数」なのでそのまま使う）
def interp_at(x_num):
    pos_X_i = np.interp(x_num, t_pos_num, pos_X, left=np.nan, right=np.nan)
    pos_Y_i = np.interp(x_num, t_pos_num, pos_Y, left=np.nan, right=np.nan)
    pos_Z_i = np.interp(x_num, t_pos_num, pos_Z, left=np.nan, right=np.nan)
    return pos_X_i, pos_Y_i, pos_Z_i

# 目盛フォーマッタ
def pos_formatter(x, pos=None):
    pos_X_i, pos_Y_i, pos_Z_i = interp_at(x)
    if np.any(~np.isfinite([pos_X_i, pos_Y_i, pos_Z_i])):
        return ""  # 範囲外は空
    return (f"{pos_X_i:0.2f}\n"
            f"{pos_Y_i:0.2f}\n"
            f"{pos_Z_i:0.2f}")

# セカンダリ x 軸（底 side）を作ってラベルを差し替え
secax = ax_7.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
secax.xaxis.set_major_formatter(mticker.FuncFormatter(pos_formatter))

# メインの時間ラベルと重ならないよう余白を広げる
ax_7.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
secax.tick_params(axis='x', which='major', pad=30)  # R/MLAT/MLTラベル

# 好みで：目盛間隔をメイン x と合わせる
secax.set_ticks(ax_7.get_xticks())

fig.text(0.01, 0.100, "hhmm", ha='center', va='center')
fig.text(0.01, 0.083, r"X-GSM", ha='center', va='center')
fig.text(0.01, 0.063, r"Y-GSM", ha='center', va='center')
fig.text(0.01, 0.045, r"Z-GSM", ha='center', va='center')

add_panel_label(ax_1, '(h)')
add_panel_label(ax_2, '(i)')
add_panel_label(ax_3, '(j)')
add_panel_label(ax_4, '(k)')
add_panel_label(ax_5, '(l)')
add_panel_label(ax_6, '(m)')
add_panel_label(ax_7, '(n)')

fig.suptitle('Arase', y=0.9)

fig.tight_layout()
plt.show()

fig.savefig(r"/mnt/j/KAW_observation/Arase_flux.pdf", bbox_inches='tight')
fig.savefig(r"/mnt/j/KAW_observation/Arase_flux.png", bbox_inches='tight')

In [ ]:
pos_da_analysis = pos_da.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

print(np.nanmin(pos_da_analysis[:, 0]), np.nanmax(pos_da_analysis[:, 0]))
print(np.nanmin(pos_da_analysis[:, 1]), np.nanmax(pos_da_analysis[:, 1]))
print(np.nanmin(pos_da_analysis[:, 2]), np.nanmax(pos_da_analysis[:, 2]))


In [ ]:
import pyspedas as psp
import pytplot as pt

time_range = ['20220901/20:00:00', '20220902/00:00:00']
psp.projects.themis.fgm(trange=time_range, probe='a', level='l2', get_support_data=True)                 # fgh: 128 Hz, fgl: 16 Hz, fgs: 2.74 sec
psp.projects.themis.efi(trange=time_range, probe='a', level='l2', datatype='efp', get_support_data=True) # efp: 512 Hz
psp.projects.themis.efi(trange=time_range, probe='a', level='l2', datatype='efi', get_support_data=True) # eff: 8 Hz

E512_data_gsm   = psp.get_data('tha_efp_gsm', xarray=True)
B16_data_gsm    = psp.get_data('tha_fgl_gsm', xarray=True)
B128_data_gsm   = psp.get_data('tha_fgh_gsm', xarray=True)

Espin_data_gsm  = psp.get_data('tha_efs_dot0_gsm', xarray=True)
Bspin_data_gsm  = psp.get_data('tha_fgs_gsm', xarray=True)

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]
E512_data_gsm   = E512_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
B16_data_gsm    = B16_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
B128_data_gsm   = B128_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')

Espin_data_gsm  = Espin_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
Bspin_data_gsm  = Bspin_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')

import xarray as xr
import numpy as np

def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

ds_E512_data_gsm = xr.Dataset(
    data_vars={
        "E512_gsm_x": (("time",), E512_data_gsm[:, 0].values),
        "E512_gsm_y": (("time",), E512_data_gsm[:, 1].values),
        "E512_gsm_z": (("time",), E512_data_gsm[:, 2].values),
    },
    coords={
        "time": E512_data_gsm["time"].values,
    },
)
ds_B16_data_gsm = xr.Dataset(
    data_vars={
        "B16_gsm_x": (("time",), B16_data_gsm[:, 0].values),
        "B16_gsm_y": (("time",), B16_data_gsm[:, 1].values),
        "B16_gsm_z": (("time",), B16_data_gsm[:, 2].values),
    },
    coords={
        "time": B16_data_gsm["time"].values,
    },
)
ds_B128_data_gsm = xr.Dataset(
    data_vars={
        "B128_gsm_x": (("time",), B128_data_gsm[:, 0].values),
        "B128_gsm_y": (("time",), B128_data_gsm[:, 1].values),
        "B128_gsm_z": (("time",), B128_data_gsm[:, 2].values),
    },
    coords={
        "time": B128_data_gsm["time"].values,
    },
)
ds_Espin_data_gsm   = xr.Dataset(
    data_vars={
        "Espin_gsm_x": (('time',), Espin_data_gsm[:, 0].values),
        "Espin_gsm_y": (('time',), Espin_data_gsm[:, 1].values),
        "Espin_gsm_z": (('time',), Espin_data_gsm[:, 2].values),
    },
    coords={
        'time': Espin_data_gsm['time'].values,
    },
)
ds_Bspin_data_gsm   = xr.Dataset(
    data_vars={
        "Bspin_gsm_x": (('time',), Bspin_data_gsm[:, 0].values),
        "Bspin_gsm_y": (('time',), Bspin_data_gsm[:, 1].values),
        "Bspin_gsm_z": (('time',), Bspin_data_gsm[:, 2].values),
    },
    coords={
        'time': Bspin_data_gsm['time'].values,
    },
)

def uniq_and_sort_time(ds):
    idx = ds.get_index('time')
    mask = ~idx.duplicated()          # 最初の出現だけ True
    return ds.isel(time=mask).sortby('time')

ds_E512_data_gsm    = uniq_and_sort_time(ds_E512_data_gsm)
ds_B16_data_gsm     = uniq_and_sort_time(ds_B16_data_gsm)
ds_B128_data_gsm    = uniq_and_sort_time(ds_B128_data_gsm)

ds_Espin_data_gsm   = uniq_and_sort_time(ds_Espin_data_gsm)
ds_Bspin_data_gsm   = uniq_and_sort_time(ds_Bspin_data_gsm)

ds_E512_data_gsm_segs = split_by_gap(ds_E512_data_gsm, gap_thr=np.timedelta64(8, 'ms'))
for ds_ in ds_E512_data_gsm_segs:
    print(ds_.time)

ds_B16_data_gsm_segs = split_by_gap(ds_B16_data_gsm, gap_thr=np.timedelta64(250, 'ms'))
for ds_ in ds_B16_data_gsm_segs:
    print(ds_.time)

ds_B128_data_gsm_segs = split_by_gap(ds_B128_data_gsm, gap_thr=np.timedelta64(250, 'ms'))
for ds_ in ds_B128_data_gsm_segs:
    print(ds_.time)

ds_Espin_data_gsm_segs = split_by_gap(ds_Espin_data_gsm, gap_thr=np.timedelta64(11, 's'))
for ds_ in ds_Espin_data_gsm_segs:
    print(ds_.time)

ds_Bspin_data_gsm_segs = split_by_gap(ds_Bspin_data_gsm, gap_thr=np.timedelta64(11, 's'))
for ds_ in ds_Bspin_data_gsm_segs:
    print(ds_.time)

def make_ds_EB_func(ds_E, E_vars, ds_B, B_vars, time_base, output_vars):
    ds_E_interp = ds_E.interp(time=time_base, method='linear')
    ds_B_interp = ds_B.interp(time=time_base, method='linear')

    da_Ex   = ds_E_interp[E_vars[0]]
    da_Ey   = ds_E_interp[E_vars[1]]
    da_Ez   = ds_E_interp[E_vars[2]]
    da_Bx   = ds_B_interp[B_vars[0]]
    da_By   = ds_B_interp[B_vars[1]]
    da_Bz   = ds_B_interp[B_vars[2]]

    ds_EB   = xr.Dataset({
        output_vars[0]: da_Ex,
        output_vars[1]: da_Ey,
        output_vars[2]: da_Ez,
        output_vars[3]: da_Bx,
        output_vars[4]: da_By,
        output_vars[5]: da_Bz,
    })

    ds_EB   = ds_EB.dropna(dim='time', how='any')

    return ds_EB

E512_vars   = ['E512_gsm_x', 'E512_gsm_y', 'E512_gsm_z']
B16_vars    = ['B16_gsm_x', 'B16_gsm_y', 'B16_gsm_z']
B128_vars   = ['B128_gsm_x', 'B128_gsm_y', 'B128_gsm_z']
EB128_vars  = ['E128_gsm_x', 'E128_gsm_y', 'E128_gsm_z', 'B128_gsm_x', 'B128_gsm_y', 'B128_gsm_z']

Espin_vars  = ['Espin_gsm_x', 'Espin_gsm_y', 'Espin_gsm_z']
Bspin_vars  = ['Bspin_gsm_x', 'Bspin_gsm_y', 'Bspin_gsm_z']
EBspin_vars = ['Espin_gsm_x', 'Espin_gsm_y', 'Espin_gsm_z', 'Bspin_gsm_x', 'Bspin_gsm_y', 'Bspin_gsm_z']

ds_EB128_gsm_segs = []

ds_EB128_gsm_segs.append(make_ds_EB_func(ds_E512_data_gsm_segs[0], E512_vars, ds_B128_data_gsm_segs[0], B128_vars, ds_B128_data_gsm_segs[0].time, EB128_vars).dropna(dim='time', how='all'))
ds_EB128_gsm_segs.append(make_ds_EB_func(ds_E512_data_gsm_segs[1], E512_vars, ds_B128_data_gsm_segs[1], B128_vars, ds_B128_data_gsm_segs[1].time, EB128_vars).dropna(dim='time', how='all'))
ds_EB128_gsm_segs.append(make_ds_EB_func(ds_E512_data_gsm_segs[2], E512_vars, ds_B128_data_gsm_segs[2], B128_vars, ds_B128_data_gsm_segs[2].time, EB128_vars).dropna(dim='time', how='all'))

ds_EBspin_gsm_segs  = []

ds_EBspin_gsm_segs.append(make_ds_EB_func(ds_Espin_data_gsm_segs[0], Espin_vars, ds_Bspin_data_gsm_segs[0], Bspin_vars, ds_Espin_data_gsm_segs[0].time, EBspin_vars).dropna(dim='time', how='all'))


psp.projects.themis.state(probe='a', trange=time_range, no_update=True)

da_THA_pos_gsm  = psp.get_data('tha_pos_gsm', xarray=True)
da_THA_pos_gsm  = da_THA_pos_gsm.sortby('time').sel(time=slice(time_range[0], time_range[1]))

da_THA_pos_unit_gsm = da_THA_pos_gsm / np.sqrt((da_THA_pos_gsm * da_THA_pos_gsm).sum(dim='v_dim'))

background_time_sec = 100 #[sec]

time_width_B_16Hz       = (B16_data_gsm.time[10] - B16_data_gsm.time[9]) / np.timedelta64(1, 's')
da_B_background         = B16_data_gsm.rolling(time=int(background_time_sec/time_width_B_16Hz), center=True).mean('time')
da_B_background_unit    = da_B_background / np.sqrt((da_B_background * da_B_background).sum(dim='v_dim'))
da_B_background_unit    = da_B_background_unit.dropna(how='all', dim='time')

time_array  = da_B_background_unit.time

da_THA_pos_unit_gsm_interp    = da_THA_pos_unit_gsm.interp(time=time_array)

da_u_   = da_THA_pos_unit_gsm_interp - (da_THA_pos_unit_gsm_interp * da_B_background_unit).sum(dim='v_dim') * da_B_background_unit

da_e_z_FAC_inGSM    = da_B_background_unit.drop_attrs()
da_e_x_FAC_inGSM    = (da_u_ / np.sqrt((da_u_ * da_u_).sum(dim='v_dim'))).drop_attrs()
da_e_y_FAC_inGSM    = (xr.apply_ufunc(np.cross, da_e_z_FAC_inGSM, da_e_x_FAC_inGSM, input_core_dims=[['v_dim'], ['v_dim']], output_core_dims=[['v_dim']], vectorize=True)).drop_attrs()

R_FAC_to_GSM = xr.concat(
    [da_e_x_FAC_inGSM, da_e_y_FAC_inGSM, da_e_z_FAC_inGSM],
    dim='axis'
)
R_FAC_to_GSM    = R_FAC_to_GSM.assign_coords(axis=['x_FAC', 'y_FAC', 'z_FAC']).assign_coords(v_dim=np.arange(3))
R_FAC_to_GSM    = R_FAC_to_GSM.dropna(dim='time', how='any')

R_GSM_to_FAC    = R_FAC_to_GSM.transpose('time', 'v_dim', 'axis')

da_e_x_GSM_inFAC    = R_GSM_to_FAC.sel(v_dim=0)
da_e_y_GSM_inFAC    = R_GSM_to_FAC.sel(v_dim=1)
da_e_z_GSM_inFAC    = R_GSM_to_FAC.sel(v_dim=2)

ds_EB128_fac_segs = []

for ds_seg in ds_EB128_gsm_segs:
    # 1) この seg の時間に合わせて回転行列を補間
    R_seg = R_GSM_to_FAC.interp(time=ds_seg.time)

    # 2) DSIベクトル (time, v_dim) を作る
    E_dsi = xr.concat(
        [ds_seg['E128_gsm_x'], ds_seg['E128_gsm_y'], ds_seg['E128_gsm_z']],
        dim='v_dim'
    )
    B_dsi = xr.concat(
        [ds_seg['B128_gsm_x'], ds_seg['B128_gsm_y'], ds_seg['B128_gsm_z']],
        dim='v_dim'
    )

    # こちらも v_dim を 0,1,2 にそろえる
    E_dsi = E_dsi.assign_coords(v_dim=np.arange(3))
    B_dsi = B_dsi.assign_coords(v_dim=np.arange(3))

    # 3) DSI → FAC 回転
    E_fac = xr.dot(E_dsi, R_seg, dims='v_dim')  # (time, axis)
    B_fac = xr.dot(B_dsi, R_seg, dims='v_dim')

    # 4) axis 次元を変数に落とす
    E_fac_ds = (
        E_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'E128_fac_x',
            'y_FAC': 'E128_fac_y',
            'z_FAC': 'E128_fac_z',
        })
    )

    B_fac_ds = (
        B_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'B128_fac_x',
            'y_FAC': 'B128_fac_y',
            'z_FAC': 'B128_fac_z',
        })
    )

    ds_fac = xr.merge([E_fac_ds, B_fac_ds]).dropna(dim='time', how='any')
    ds_fac = ds_fac.assign_attrs(ds_seg.attrs)
    ds_EB128_fac_segs.append(ds_fac)

ds_EBspin_fac_segs  = []

for ds_seg in ds_EBspin_gsm_segs:
    # 1) この seg の時間に合わせて回転行列を補間
    R_seg = R_GSM_to_FAC.interp(time=ds_seg.time)

    # 2) DSIベクトル (time, v_dim) を作る
    E_dsi = xr.concat(
        [ds_seg['Espin_gsm_x'], ds_seg['Espin_gsm_y'], ds_seg['Espin_gsm_z']],
        dim='v_dim'
    )
    B_dsi = xr.concat(
        [ds_seg['Bspin_gsm_x'], ds_seg['Bspin_gsm_y'], ds_seg['Bspin_gsm_z']],
        dim='v_dim'
    )

    # こちらも v_dim を 0,1,2 にそろえる
    E_dsi = E_dsi.assign_coords(v_dim=np.arange(3))
    B_dsi = B_dsi.assign_coords(v_dim=np.arange(3))

    # 3) DSI → FAC 回転
    E_fac = xr.dot(E_dsi, R_seg, dims='v_dim')  # (time, axis)
    B_fac = xr.dot(B_dsi, R_seg, dims='v_dim')

    # 4) axis 次元を変数に落とす
    E_fac_ds = (
        E_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'Espin_fac_x',
            'y_FAC': 'Espin_fac_y',
            'z_FAC': 'Espin_fac_z',
        })
    )

    B_fac_ds = (
        B_fac
        .to_dataset(dim='axis')
        .rename({
            'x_FAC': 'Bspin_fac_x',
            'y_FAC': 'Bspin_fac_y',
            'z_FAC': 'Bspin_fac_z',
        })
    )

    ds_fac = xr.merge([E_fac_ds, B_fac_ds]).dropna(dim='time', how='any')
    ds_fac = ds_fac.assign_attrs(ds_seg.attrs)
    ds_EBspin_fac_segs.append(ds_fac)

In [ ]:
Ex_fac_vec = xr.concat([ds_EB128_fac_segs[0]['E128_fac_x'], ds_EB128_fac_segs[1]['E128_fac_x'], ds_EB128_fac_segs[2]['E128_fac_x']], dim='time')
Ey_fac_vec = xr.concat([ds_EB128_fac_segs[0]['E128_fac_y'], ds_EB128_fac_segs[1]['E128_fac_y'], ds_EB128_fac_segs[2]['E128_fac_y']], dim='time')
Bx_fac_vec = xr.concat([ds_EB128_fac_segs[0]['B128_fac_x'], ds_EB128_fac_segs[1]['B128_fac_x'], ds_EB128_fac_segs[2]['B128_fac_x']], dim='time')
By_fac_vec = xr.concat([ds_EB128_fac_segs[0]['B128_fac_y'], ds_EB128_fac_segs[1]['B128_fac_y'], ds_EB128_fac_segs[2]['B128_fac_y']], dim='time')

In [ ]:
Ex_fac_spin = ds_EBspin_fac_segs[0]['Espin_fac_x']
Ey_fac_spin = ds_EBspin_fac_segs[0]['Espin_fac_y']
Bx_fac_spin = ds_EBspin_fac_segs[0]['Bspin_fac_x']
By_fac_spin = ds_EBspin_fac_segs[0]['Bspin_fac_y']

In [ ]:
Eperp_2_THA = ((Ex_fac_vec)**2E0 + (Ey_fac_vec)**2E0)
Bperp_2_THA = ((Bx_fac_vec)**2E0 + (By_fac_vec)**2E0)

mu_0 = 4.*np.pi * 1E-7

S_para_THA = (Ex_fac_vec * By_fac_vec - Ey_fac_vec * Bx_fac_vec) / mu_0 * 1E-12 #W/m^2

print(Eperp_2_THA)
print(Bperp_2_THA)
print(S_para_THA)

In [ ]:
Eperp_2_spin_THA    = ((Ex_fac_spin)**2E0 + (Ey_fac_spin)**2E0)
Bperp_2_spin_THA    = ((Bx_fac_spin)**2E0 + (By_fac_spin)**2E0)

mu_0 = 4.*np.pi * 1E-7

S_para_spin_THA = (Ex_fac_spin * By_fac_spin - Ey_fac_spin * Bx_fac_spin) / mu_0 * 1E-12 #W/m^2

print(Eperp_2_spin_THA)
print(Bperp_2_spin_THA)
print(S_para_spin_THA)

In [ ]:
psp.projects.themis.state(trange=trange, probe='a')

In [ ]:
psp.cotrans(name_in='tha_pos_gsm', name_out='tha_pos_sm', coord_in='gsm', coord_out='sm')
THA_SM_pos = psp.get_data('tha_pos_sm', xarray=True).sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
print(THA_SM_pos)

THA_rmlatmlt_R = np.sqrt(THA_SM_pos.data[:, 0]**2E0 + THA_SM_pos.data[:, 1]**2E0 + THA_SM_pos.data[:, 2]**2E0) / 6378.1
THA_rmlatmlt_MLAT = np.rad2deg(np.arctan2(THA_SM_pos.data[:, 2], np.sqrt(THA_SM_pos.data[:, 0]**2E0 + THA_SM_pos.data[:, 1]**2E0)))
THA_rmlatmlt_MLT = np.rad2deg(np.arctan2(THA_SM_pos.data[:, 1], THA_SM_pos.data[:, 0])) / 15. + 12.

THA_rmlatmlt_L  = THA_rmlatmlt_R / np.cos(np.deg2rad(THA_rmlatmlt_MLAT))**2E0

print(np.nanmax(THA_rmlatmlt_R), np.nanmin(THA_rmlatmlt_R))
print(np.nanmax(THA_rmlatmlt_MLAT), np.nanmin(THA_rmlatmlt_MLAT))
print(np.nanmax(THA_rmlatmlt_MLT), np.nanmin(THA_rmlatmlt_MLT))

print(np.nanmax(THA_rmlatmlt_L), np.nanmin(THA_rmlatmlt_L))

In [ ]:
import matplotlib.ticker as mticker

fig = plt.figure(figsize=(10, 22))
gs = fig.add_gridspec(7, 2, width_ratios=[1, 0.025], wspace=0.05, hspace=0.15)
ax_1 = fig.add_subplot(gs[0, 0])
cax_1 = fig.add_subplot(gs[0, 1])
ax_2 = fig.add_subplot(gs[1, 0], sharex=ax_1)
cax_2 = fig.add_subplot(gs[1, 1])
ax_3 = fig.add_subplot(gs[2, 0], sharex=ax_1)
cax_3 = fig.add_subplot(gs[2, 1])
ax_4 = fig.add_subplot(gs[3, 0], sharex=ax_1)
cax_4 = fig.add_subplot(gs[3, 1])
ax_5 = fig.add_subplot(gs[4, 0], sharex=ax_1)
ax_6 = fig.add_subplot(gs[5, 0], sharex=ax_1)
ax_7 = fig.add_subplot(gs[6, 0], sharex=ax_1)

ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)
ax_3.tick_params(axis='x', which='both', labelbottom=False)
ax_4.tick_params(axis='x', which='both', labelbottom=False)
ax_5.tick_params(axis='x', which='both', labelbottom=False)
ax_6.tick_params(axis='x', which='both', labelbottom=False)

ytitle_proton = r'ESA ion' + '\n' + r'[eV]'
ztitle_proton = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_THEMIS_A(fig, ax_1, cax_1, dn_tha_proton, np.nanpercentile(dn_tha_proton.diff_number_flux, 25), np.nanpercentile(dn_tha_proton.diff_number_flux, 99), 4E1, 1E4, ytitle_proton, ztitle_proton)

ytitle_proton_pa = r'ESA ion' + '\n' + r'[deg]'
ztitle_proton_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_THEMIS_A(fig, ax_2, cax_2, da_tha_proton_pa, np.nanpercentile(da_tha_proton_pa.diff_number_flux, 10), np.nanpercentile(da_tha_proton_pa.diff_number_flux, 99), ytitle_proton_pa, ztitle_proton_pa)

ytitle_electron = r'ESA $\mathrm{e}^{-}$' + '\n' + r'[eV]'
ztitle_electron = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_THEMIS_A(fig, ax_3, cax_3, dn_tha_electron, np.nanpercentile(dn_tha_electron.diff_number_flux, 25), np.nanpercentile(dn_tha_electron.diff_number_flux, 99), 4E1, 1E4, ytitle_electron, ztitle_electron)

ytitle_electron_pa = r'ESA $\mathrm{e}^{-}$' + '\n' + r'[deg]'
ztitle_electron_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_THEMIS_A(fig, ax_4, cax_4, da_tha_electron_pa, np.nanpercentile(da_tha_electron_pa.diff_number_flux, 10), np.nanpercentile(da_tha_electron_pa.diff_number_flux, 99), ytitle_electron_pa, ztitle_electron_pa)

ax_5.scatter(Eperp_2_THA.time, np.sqrt(Eperp_2_THA.data), c='k', s=0.1)
ax_5.plot(Eperp_2_spin_THA.time, np.sqrt(Eperp_2_spin_THA.data), c='b', lw=1)
ax_5.set_ylabel(r'$|\mathbf{E}_{\perp}|$' + '\n' + r'[$\mathrm{mV/m}$]')
ax_5.set_ylim(ymin=0)#, ymax=4)
ax_5.minorticks_on()
ax_5.grid(which='both', alpha=0.5)

ax_6.scatter(Bperp_2_THA.time, np.sqrt(Bperp_2_THA.data), c='k', s=0.1)
ax_6.plot(Bperp_2_spin_THA.time, np.sqrt(Bperp_2_spin_THA.data), c='b', lw=1)
ax_6.set_ylabel(r'$|\mathbf{B}_{\perp}|$' + '\n' + r'[$\mathrm{nT}$]')
ax_6.set_ylim(ymin=0)
ax_6.minorticks_on()
ax_6.grid(which='both', alpha=0.5)


ax_7.scatter(S_para_THA.time, S_para_THA.data*1E3, c='k', s=0.1)
ax_7.plot(S_para_spin_THA.time, S_para_spin_THA.data*1E3, c='b', lw=1)
ax_7.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
ax_7.minorticks_on()
ax_7.grid(which='both', alpha=0.5)


time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_7.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
ax_7.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax_7.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))

def add_panel_label(ax, label, x=-0.15, y=0.90):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

def to_py_datetime(t_np64):
    return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)

# 軌道データ
THA_GSM_pos = psp.get_data('tha_pos_gsm', xarray=True).sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
t_pos_py = to_py_datetime(THA_GSM_pos.time.values)
t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数

earth_radius = 6378.1  # km
pos_X   = np.asarray(THA_GSM_pos[:, 0] / earth_radius, dtype=float)
pos_Y   = np.asarray(THA_GSM_pos[:, 1] / earth_radius, dtype=float)
pos_Z   = np.asarray(THA_GSM_pos[:, 2] / earth_radius, dtype=float)

# 補間関数（tick の x は「日数」なのでそのまま使う）
def interp_at(x_num):
    pos_X_i = np.interp(x_num, t_pos_num, pos_X, left=np.nan, right=np.nan)
    pos_Y_i = np.interp(x_num, t_pos_num, pos_Y, left=np.nan, right=np.nan)
    pos_Z_i = np.interp(x_num, t_pos_num, pos_Z, left=np.nan, right=np.nan)
    return pos_X_i, pos_Y_i, pos_Z_i

# 目盛フォーマッタ
def pos_formatter(x, pos=None):
    pos_X_i, pos_Y_i, pos_Z_i = interp_at(x)
    if np.any(~np.isfinite([pos_X_i, pos_Y_i, pos_Z_i])):
        return ""  # 範囲外は空
    return (f"{pos_X_i:0.2f}\n"
            f"{pos_Y_i:0.2f}\n"
            f"{pos_Z_i:0.2f}")

# セカンダリ x 軸（底 side）を作ってラベルを差し替え
secax = ax_7.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
secax.xaxis.set_major_formatter(mticker.FuncFormatter(pos_formatter))

# メインの時間ラベルと重ならないよう余白を広げる
ax_7.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
secax.tick_params(axis='x', which='major', pad=30)  # R/MLAT/MLTラベル

# 好みで：目盛間隔をメイン x と合わせる
secax.set_ticks(ax_7.get_xticks())

fig.text(0.01, 0.100, "hhmm", ha='center', va='center')
fig.text(0.01, 0.083, r"X-GSM", ha='center', va='center')
fig.text(0.01, 0.063, r"Y-GSM", ha='center', va='center')
fig.text(0.01, 0.045, r"Z-GSM", ha='center', va='center')

add_panel_label(ax_1, '(a)')
add_panel_label(ax_2, '(b)')
add_panel_label(ax_3, '(c)')
add_panel_label(ax_4, '(d)')
add_panel_label(ax_5, '(e)')
add_panel_label(ax_6, '(f)')
add_panel_label(ax_7, '(g)')

fig.suptitle('THEMIS-A', y=0.9)

fig.tight_layout()
plt.show()

fig.savefig(r"/mnt/j/KAW_observation/THEMIS-A_flux.pdf", bbox_inches='tight')
fig.savefig(r"/mnt/j/KAW_observation/THEMIS-A_flux.png", bbox_inches='tight')

# SYM-H index, AE indexのplot

In [ ]:
psp.projects.omni.data(trange=time_range)
print(psp.tplot_names())

In [ ]:
da_sym_h    = psp.get_data('SYM_H', xarray=True)
da_ae_index = psp.get_data('AE_INDEX', xarray=True)
da_au_index = psp.get_data('AU_INDEX', xarray=True)
da_al_index = psp.get_data('AL_INDEX', xarray=True)

print(da_sym_h)
print(da_ae_index)
print(da_au_index)
print(da_al_index)

In [ ]:
from datetime import datetime

time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

da_sym_h_analysis       = da_sym_h.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_ae_index_analysis    = da_ae_index.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_au_index_analysis    = da_au_index.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_al_index_analysis    = da_al_index.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

fig = plt.figure(figsize=(10, 10))

gs      = fig.add_gridspec(3, 1)
ax_0    = fig.add_subplot(gs[0, 0])
ax_1    = fig.add_subplot(gs[1:, 0])

ax_0.tick_params(axis='x', which='both', labelbottom=False)

ax_0.plot(da_sym_h_analysis.time,       da_sym_h_analysis.data,     lw=1, c='k')
ax_1.plot(da_ae_index_analysis.time,    da_ae_index_analysis.data,  lw=1, c='r',        label='AE')
ax_1.plot(da_au_index_analysis.time,    da_au_index_analysis.data,  lw=1, c='b',        label='AU')
ax_1.plot(da_al_index_analysis.time,    da_al_index_analysis.data,  lw=1, c='green',    label='AL')

ax_0.set_ylabel(r'SYM-H index'          + '\n' + '[nT]')
ax_1.set_ylabel(r'AE, AU, AL indices'   + '\n' + '[nT]')

ax_0.minorticks_on()
ax_0.grid(which='both', alpha=0.5)
ax_1.minorticks_on()
ax_1.grid(which='both', alpha=0.5)

ax_1.legend(ncol=3, fontsize=15)

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_1.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)

fig.tight_layout()

plt.show(fig)

In [ ]:
from datetime import datetime

time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

da_sym_h_analysis       = da_sym_h.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_ae_index_analysis    = da_ae_index.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_au_index_analysis    = da_au_index.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_al_index_analysis    = da_al_index.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

fig = plt.figure(figsize=(11, 11))

gs      = fig.add_gridspec(1, 1)
ax_0    = fig.add_subplot(gs[0, 0])

#ax_0.tick_params(axis='x', which='both', labelbottom=False)

ax_0.plot(da_ae_index_analysis.time,    da_ae_index_analysis.data,  lw=1, c='r',        label='AE')
ax_0.plot(da_au_index_analysis.time,    da_au_index_analysis.data,  lw=1, c='b',        label='AU')
ax_0.plot(da_al_index_analysis.time,    da_al_index_analysis.data,  lw=1, c='green',    label='AL')

ax_0.set_ylabel(r'AE, AU, AL indices'   + '\n' + '[nT]')

ax_0.minorticks_on()
ax_0.grid(which='both', alpha=0.5)

ax_0.legend(ncol=3)

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_0.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
ax_0.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax_0.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))

add_panel_label(ax_0, '(d)', -0.05)

fig.tight_layout()

plt.show(fig)
fig.savefig(r"/mnt/j/KAW_observation/Figure_1_b.pdf", bbox_inches='tight')
fig.savefig(r"/mnt/j/KAW_observation/Figure_1_b.png", bbox_inches='tight')

# HFAのplot

In [ ]:
ergpy.pwe_hfa(trange=time_range, level='l2')
ergpy.pwe_hfa(trange=time_range, level='l3')
ergpy.mgf(trange=time_range, level='l2', datatype='64hz')

In [ ]:
B_total = np.sqrt(psp.get_data('erg_mgf_l2_mag_64hz_dsi', xarray=True).isel(v_dim=0)**2E0 + psp.get_data('erg_mgf_l2_mag_64hz_dsi', xarray=True).isel(v_dim=1)**2E0 + psp.get_data('erg_mgf_l2_mag_64hz_dsi', xarray=True).isel(v_dim=2)**2E0)
dt = (B_total.time[1] - B_total.time[0]) / np.timedelta64(1, 's')
B_total = B_total.rolling(time=int(100 / dt), center=True).mean('time')

ep_0 = 8.8541878188E-12 #[A^2 s^4 / kg / m^3]
m_e  = 9.1093837E-31    #[kg]
elementary_charge = 1.60217663E-19  #[A s]

n_e = psp.get_data('erg_pwe_hfa_l3_1min_ne_mgf', xarray=True).interp(time=B_total.time, method='linear')

f_ce = elementary_charge * B_total*1E-9 / m_e / 2 / np.pi / 1E3                     # [kHz]
f_pe = np.sqrt(n_e*1E6 * elementary_charge**2E0 / m_e / ep_0) / 2 / np.pi / 1E3     # [kHz]

f_UHR = np.sqrt(f_ce**2E0 + f_pe**2E0)  # [kHz]

da_f_UHR    = xr.DataArray(data=f_UHR, dims=['time'], coords={'time': B_total.time}, name='f_UHR')
da_f_ce     = xr.DataArray(data=f_ce,  dims=['time'], coords={'time': B_total.time}, name='f_ce')
da_f_pe     = xr.DataArray(data=f_pe,  dims=['time'], coords={'time': B_total.time}, name='f_pe')

In [ ]:
da_hfa_spectral = psp.get_data('erg_pwe_hfa_l2_low_spectra_esum', xarray=True)
da_hfa_spectral

In [ ]:
from datetime import datetime
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm


time_range_analysis     = ['20220901/22:00:00', '20220902/00:00:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

da_hfa_spectral_analysis    = da_hfa_spectral.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_f_UHR_analysis           = da_f_UHR.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_f_ce_analysis            = da_f_ce.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_f_pe_analysis            = da_f_pe.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

fig = plt.figure(figsize=(12, 5))

ax_0    = fig.add_subplot(111)

T_0 = mdates.date2num(da_hfa_spectral_analysis.time.values)
F_0 = da_hfa_spectral_analysis.spec_bins.values
Z_0 = da_hfa_spectral_analysis.values.astype(float)

T_0_m = np.tile(T_0, (F_0.size, 1)).T
F_0_m = np.tile(F_0, (T_0.size, 1))

pcm = ax_0.pcolormesh(T_0_m, F_0_m, Z_0, shading='auto', norm=LogNorm(vmin=1E-8, vmax=1E-4), cmap='turbo')
ax_0.plot(da_f_UHR_analysis.time, da_f_UHR_analysis.data, c='white', lw=2)
ax_0.minorticks_on()
ax_0.set_yscale('log')
ax_0.set_ylim(5E0, 5E1)
ax_0.set_ylabel('PWE-HFA [kHz]')
ax_0.grid(which='both', alpha=0.5)

#ax_1.legend(ncol=3, fontsize=15)

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_0.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)

cax = ax_0.inset_axes([1.01, 0.05, 0.02, 0.9])
cb = plt.colorbar(pcm, cax=cax)
cb.minorticks_on()
cb.set_label(r'$[\mathrm{(mV/m)}^{2} / \mathrm{Hz}]$')

fig.tight_layout()

plt.show(fig)